# 03 · Universe selection & overfitting rejection

**Story.** Narrowing the universe to the in-sample best subset almost always lifts the backtest Sharpe — and almost always overfits. The disciplined test is whether the wider universe carries *equal signal quality* (information coefficient). If it does, you take the wider, more robust universe even at a lower in-sample Sharpe.

> Synthetic; illustrative. The real decision was to run the **12-fund** universe (Sharpe ~1.16) over the in-sample-optimal **7-fund** one (~1.56), because the extra funds' IC (+0.33) matched the keepers' (+0.32) — README.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import qdii_fve

ROOT = Path(qdii_fve.__file__).resolve().parents[2]
SAMPLE = ROOT / "data" / "sample"

bars = pd.read_csv(SAMPLE / "SYNTHETIC_daily_bars.csv", parse_dates=["date"])
nav = pd.read_csv(SAMPLE / "SYNTHETIC_nav.csv", parse_dates=["date"])
und = pd.read_csv(SAMPLE / "SYNTHETIC_ndx_underlying.csv", parse_dates=["date"]).set_index("date")["close"]
fx = pd.read_csv(SAMPLE / "SYNTHETIC_usdcnh.csv", parse_dates=["date"]).set_index("date")

close = bars.pivot(index="date", columns="etf_id", values="close")
bid = bars.pivot(index="date", columns="etf_id", values="bid")
ask = bars.pivot(index="date", columns="etf_id", values="ask")
nav_pub = nav.pivot(index="date", columns="etf_id", values="nav").reindex(close.index)
print("SYNTHETIC sample — illustrative only, not the real figures.")
print(f"{close.shape[1]} funds, {close.shape[0]} days")

In [ ]:
from qdii_fve.fair_value.reconstruct import reconstruct_series

# NAV here is one session lagged, so its strike date is the previous day: scale
# the published NAV by the underlying and USDCNH move since then. (The library's
# roll_forward_nav handles arbitrary vintages; this sample uses a 1-day lag.)
u_prev = und.shift(1)
fx_prev = fx["usdcnh"].shift(1)
fair_value = pd.DataFrame(index=close.index, columns=close.columns, dtype=float)
for f in close.columns:
    fair_value[f] = reconstruct_series(
        nav_value=nav_pub[f],
        nav_underlying=u_prev,
        nav_fx=fx_prev,
        current_underlying=und,
        current_fx=fx["usdcnh"],
    )
fair_value.head()

## Signal quality: information coefficient

IC = mean daily cross-sectional rank correlation between cheapness (−dev) and the next day's return. We compare the IC of the *full* universe with an in-sample-narrowed subset.

In [ ]:
from qdii_fve.backtest.signal import cross_sectional_dev, observed_premium_bps
from qdii_fve.backtest.engine import run_rotation
from qdii_fve.backtest.universe import annualized_sharpe, forward_returns, rank_ic
from qdii_fve.config import load_config

cfg = load_config()
premium_bps = observed_premium_bps(close, fair_value)
dev = cross_sectional_dev(premium_bps, lookback=cfg.signal.norm_lookback_days)
fwd = forward_returns(close, horizon=1)


def rotation_sharpe(universe):
    c = close[universe]
    res = run_rotation(dev[universe], c, c, c,
                       threshold_bps=cfg.signal.switch_threshold_bps, fill_delay=1)
    return annualized_sharpe(res.daily_excess)


# What an overfitter does: greedily drop the fund whose removal most raises the
# in-sample Sharpe, until 7 remain. This maximises the backtest, by construction.
full_universe = list(close.columns)
narrow = list(full_universe)
while len(narrow) > 7:
    scored = [(rotation_sharpe([x for x in narrow if x != f]), f) for f in narrow]
    _, drop = max(scored)
    narrow = [x for x in narrow if x != drop]
print("full  :", len(full_universe), "funds")
print("narrow:", narrow, "(greedily chosen to maximise in-sample Sharpe)")

In [ ]:
summary = pd.DataFrame(
    {"universe": ["full (all funds)", "narrow (in-sample best 7)"],
     "in_sample_sharpe": [rotation_sharpe(full_universe), rotation_sharpe(narrow)],
     "rank_IC": [rank_ic(dev[full_universe], fwd[full_universe]),
                 rank_ic(dev[narrow], fwd[narrow])]}
)
summary

## The discipline

The narrowed universe posts a higher **in-sample** Sharpe — that is what overfitting looks like. The decision rule is not "take the higher Sharpe"; it is "take the wider universe unless its signal quality is genuinely worse". On the real data the ICs were essentially equal (**+0.33 vs +0.32**), so the 12-fund universe was deployed despite the 7-fund's higher in-sample Sharpe (**1.16 vs 1.56**). Choosing robustness over the in-sample maximum is the point.

This mirrors a separate result in the research: re-sweeping the rotation parameters in the deployment's own frame showed the higher-scoring cells were knife-edge (good numbers that did not extend to their neighbours), i.e. overfit — so the deployed parameters were kept.